<a href="https://colab.research.google.com/github/Lddprogram/Lddcan/blob/main/lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤗 x 🦾: Training ACT with LeRobot Notebook

Welcome to the **LeRobot ACT training notebook**! This notebook provides a ready-to-run setup for training imitation learning policies using the [🤗 LeRobot](https://github.com/huggingface/lerobot) library.

In this example, we train an `ACT` policy using a dataset hosted on the [Hugging Face Hub](https://huggingface.co/), and optionally track training metrics with [Weights & Biases (wandb)](https://wandb.ai/).

## ⚙️ Requirements
- A Hugging Face dataset repo ID containing your training data (`--dataset.repo_id=YOUR_USERNAME/YOUR_DATASET`)
- Optional: A [wandb](https://wandb.ai/) account if you want to enable training visualization
- Recommended: GPU runtime (e.g., NVIDIA A100) for faster training

## ⏱️ Expected Training Time
Training with the `ACT` policy for 100,000 steps typically takes **about 1.5 hours on an NVIDIA A100** GPU. On less powerful GPUs or CPUs, training may take significantly longer.

## Example Output
Model checkpoints, logs, and training plots will be saved to the specified `--output_dir`. If `wandb` is enabled, progress will also be visualized in your wandb project dashboard.


## Install conda
This cell uses `condacolab` to bootstrap a full Conda environment inside Google Colab.


In [1]:
!pip install -q condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


## Install LeRobot
This cell clones the `lerobot` repository from Hugging Face, installs FFmpeg (version 7.1.1), and installs the package in editable mode.


In [3]:

!git clone https://github.com/huggingface/lerobot.git
%cd /content/lerobot
!git checkout v0.4.4
!conda install ffmpeg=7.1.1 -c conda-forge -y
!pip install -e ".[pusht]"

Cloning into 'lerobot'...
remote: Enumerating objects: 44936, done.
remote: Counting objects: 100% (498/498), done.
remote: Compressing objects: 100% (201/201), done.
remote: Total 44936 (delta 398), reused 311 (delta 293), pack-reused 44438 (from 4)
Receiving objects: 100% (44936/44936), 237.58 MiB | 41.16 MiB/s, done.
Resolving deltas: 100% (28366/28366), done.
/content/lerobot
fatal: not a git repository (or any of the parent directories): .git
Channels:
 - conda-forge
Platform: linux-64
Solving environment: | / - \ | done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 26.3.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



# All requested packages already installed.

Obtaining file:///content/lerobot
ERROR: file:///content/lerobot does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


## Weights & Biases login
This cell logs you into Weights & Biases (wandb) to enable experiment tracking and logging.

In [ ]:
!wandb login

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 


Aborted!


## Start training ACT with LeRobot

This cell runs the `train.py` script from the `lerobot` library to train a robot control policy.  

Make sure to adjust the following arguments to your setup:

1. `--dataset.repo_id=YOUR_HF_USERNAME/YOUR_DATASET`:  
   Replace this with the Hugging Face Hub repo ID where your dataset is stored, e.g., `pepijn223/il_gym0`.

2. `--policy.type=act`:  
   Specifies the policy configuration to use. `act` refers to [configuration_act.py](../lerobot/common/policies/act/configuration_act.py), which will automatically adapt to your dataset’s setup (e.g., number of motors and cameras).

3. `--output_dir=outputs/train/...`:  
   Directory where training logs and model checkpoints will be saved.

4. `--job_name=...`:  
   A name for this training job, used for logging and Weights & Biases.

5. `--policy.device=cuda`:  
   Use `cuda` if training on an NVIDIA GPU. Use `mps` for Apple Silicon, or `cpu` if no GPU is available.

6. `--wandb.enable=true`:  
   Enables Weights & Biases for visualizing training progress. You must be logged in via `wandb login` before running this. Set to `False` if you do not plan on using Weights & Biases.

重新挂载Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p /content/lerobot/outputs/train
!cp -r /content/drive/MyDrive/pusht_act_backup /content/lerobot/outputs/train/pusht_act

In [ ]:
# #自动备份，重新训练
# from google.colab import drive
# drive.mount('/content/drive')
# import os
# import shutil
# import threading
# import time

# LOCAL_OUTPUT_DIR = "/content/lerobot/outputs/train/pusht_act"
# DRIVE_BACKUP_DIR = "/content/drive/MyDrive/pusht_act_backup"

# os.makedirs("/content/lerobot/outputs/train", exist_ok=True)
# os.makedirs("/content/drive/MyDrive", exist_ok=True)

# def backup_loop(interval_seconds=300):
#     while True:
#         try:
#             if os.path.exists(LOCAL_OUTPUT_DIR) and len(os.listdir(LOCAL_OUTPUT_DIR)) > 0:
#                 tmp_dir = DRIVE_BACKUP_DIR + "_tmp"

#                 if os.path.exists(tmp_dir):
#                     shutil.rmtree(tmp_dir)

#                 shutil.copytree(LOCAL_OUTPUT_DIR, tmp_dir)

#                 if os.path.exists(DRIVE_BACKUP_DIR):
#                     shutil.rmtree(DRIVE_BACKUP_DIR)
#                 os.rename(tmp_dir, DRIVE_BACKUP_DIR)

#                 print(f"[自动备份] 已同步到 Drive: {time.strftime('%Y-%m-%d %H:%M:%S')}")
#         except Exception as e:
#             print("[自动备份失败]", e)

#         time.sleep(interval_seconds)

# backup_thread = threading.Thread(
#     target=backup_loop,
#     kwargs={"interval_seconds": 300},
#     daemon=True
# )
# backup_thread.start()

# print("自动备份线程已启动：每 5 分钟备份一次")

Mounted at /content/drive
自动备份线程已启动：每 5 分钟备份一次


In [ ]:
#删除last目录，防止发生冲突
!ls -l /content/lerobot/outputs/train/pusht_act/checkpoints
!rm -rf /content/lerobot/outputs/train/pusht_act/checkpoints/last


ls: cannot access '/content/lerobot/outputs/train/pusht_act/checkpoints': No such file or directory


In [ ]:
# #继续训练，继承原checkpoints
# import os
# from datetime import datetime

# # 改成你自己的 Hugging Face 用户名
# os.environ["HF_USER"] = "LDD11"

# # 原始训练目录（保留不动）
# OLD_LOCAL_DIR = "/content/lerobot/outputs/train/pusht_act"
# OLD_DRIVE_DIR = "/content/drive/MyDrive/pusht_act_backup"

# # 这次续训的新分支名字
# RUN_NAME = "pusht_act_resume_" + datetime.now().strftime("%Y%m%d_%H%M%S")

# # 新的本地训练目录与新的 Drive 备份目录
# NEW_LOCAL_DIR = f"/content/lerobot/outputs/train/{RUN_NAME}"
# NEW_DRIVE_DIR = f"/content/drive/MyDrive/{RUN_NAME}_backup"

# print("HF_USER =", os.environ["HF_USER"])
# print("OLD_LOCAL_DIR =", OLD_LOCAL_DIR)
# print("OLD_DRIVE_DIR =", OLD_DRIVE_DIR)
# print("RUN_NAME =", RUN_NAME)
# print("NEW_LOCAL_DIR =", NEW_LOCAL_DIR)
# print("NEW_DRIVE_DIR =", NEW_DRIVE_DIR)

HF_USER = LDD11
OLD_LOCAL_DIR = /content/lerobot/outputs/train/pusht_act
OLD_DRIVE_DIR = /content/drive/MyDrive/pusht_act_backup
RUN_NAME = pusht_act_resume_20260408_024233
NEW_LOCAL_DIR = /content/lerobot/outputs/train/pusht_act_resume_20260408_024233
NEW_DRIVE_DIR = /content/drive/MyDrive/pusht_act_resume_20260408_024233_backup


In [ ]:
# !mkdir -p /content/lerobot/outputs/train

# if not os.path.exists(OLD_LOCAL_DIR):
#     !cp -r /content/drive/MyDrive/pusht_act_backup /content/lerobot/outputs/train/pusht_act

# print("原始训练目录检查完成：")
# !ls -R /content/lerobot/outputs/train/pusht_act | head -200

原始训练目录检查完成：
/content/lerobot/outputs/train/pusht_act:
checkpoints

/content/lerobot/outputs/train/pusht_act/checkpoints:
020000
040000

/content/lerobot/outputs/train/pusht_act/checkpoints/020000:
pretrained_model
training_state

/content/lerobot/outputs/train/pusht_act/checkpoints/020000/pretrained_model:
config.json
model.safetensors
policy_postprocessor.json
policy_postprocessor_step_0_unnormalizer_processor.safetensors
policy_preprocessor.json
policy_preprocessor_step_3_normalizer_processor.safetensors
train_config.json

/content/lerobot/outputs/train/pusht_act/checkpoints/020000/training_state:
optimizer_param_groups.json
optimizer_state.safetensors
rng_state.safetensors
training_step.json

/content/lerobot/outputs/train/pusht_act/checkpoints/040000:
pretrained_model
training_state

/content/lerobot/outputs/train/pusht_act/checkpoints/040000/pretrained_model:
config.json
model.safetensors
policy_postprocessor.json
policy_postprocessor_step_0_unnormalizer_processor.safetensors
poli

In [ ]:
# import os

# old_resume_cfg = os.path.join(
#     OLD_LOCAL_DIR,
#     "checkpoints/last/pretrained_model/train_config.json"
# )

# if not os.path.exists(OLD_LOCAL_DIR):
#     raise FileNotFoundError(f"旧本地训练目录不存在：{OLD_LOCAL_DIR}")

# if not os.path.exists(old_resume_cfg):
#     raise FileNotFoundError(f"旧目录中缺少恢复配置文件：{old_resume_cfg}")

# if os.path.exists(NEW_LOCAL_DIR):
#     raise FileExistsError(f"新本地目录已存在，停止以避免覆盖：{NEW_LOCAL_DIR}")

# if os.path.exists(NEW_DRIVE_DIR):
#     raise FileExistsError(f"新 Drive 备份目录已存在，停止以避免覆盖：{NEW_DRIVE_DIR}")

# print("检查通过：原记录可用，新目录不存在，不会覆盖旧记录。")

FileNotFoundError: 旧目录中缺少恢复配置文件：/content/lerobot/outputs/train/pusht_act/checkpoints/last/pretrained_model/train_config.json

In [ ]:
# import shutil

# shutil.copytree(OLD_LOCAL_DIR, NEW_LOCAL_DIR)

# print("已复制原记录到新分支目录：")
# print(NEW_LOCAL_DIR)

# !ls -l {NEW_LOCAL_DIR}/checkpoints/last/pretrained_model/train_config.json

已复制原记录到新分支目录：
/content/lerobot/outputs/train/pusht_act_resume_20260408_024233
ls: cannot access '/content/lerobot/outputs/train/pusht_act_resume_20260408_024233/checkpoints/last/pretrained_model/train_config.json': No such file or directory


In [ ]:
#自动备份
import os
import shutil
import threading
import time

LOCAL_OUTPUT_DIR = "/content/lerobot/outputs/train/pusht_act"
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/pusht_act_resume_backup_re1"

def has_numeric_checkpoint(checkpoint_root):
    if not os.path.exists(checkpoint_root):
        return False
    for name in os.listdir(checkpoint_root):
        full_path = os.path.join(checkpoint_root, name)
        if os.path.isdir(full_path) and name.isdigit():
            return True
    return False

def backup_loop(interval_seconds=300):
    while True:
        try:
            checkpoint_root = os.path.join(LOCAL_OUTPUT_DIR, "checkpoints")

            if os.path.exists(LOCAL_OUTPUT_DIR) and has_numeric_checkpoint(checkpoint_root):
                tmp_dir = DRIVE_BACKUP_DIR + "_tmp"

                if os.path.exists(tmp_dir):
                    shutil.rmtree(tmp_dir)

                shutil.copytree(LOCAL_OUTPUT_DIR, tmp_dir)

                if os.path.exists(DRIVE_BACKUP_DIR):
                    shutil.rmtree(DRIVE_BACKUP_DIR)

                os.rename(tmp_dir, DRIVE_BACKUP_DIR)
                print(f"[自动备份] 已同步到 Drive: {time.strftime('%Y-%m-%d %H:%M:%S')}")
            else:
                print("[自动备份] 当前训练输出目录或有效 checkpoint 不存在，本轮跳过")

        except Exception as e:
            print("[自动备份失败]", e)

        time.sleep(interval_seconds)

backup_thread = threading.Thread(
    target=backup_loop,
    kwargs={"interval_seconds": 300},
    daemon=True
)
backup_thread.start()

print("自动备份线程已启动")

自动备份线程已启动


In [ ]:
# #继续训练
# # 从新分支目录继续训练（不会覆盖原记录）
# !cd /content/lerobot && python src/lerobot/scripts/lerobot_train.py \
#   --config_path={NEW_LOCAL_DIR}/checkpoints/last/pretrained_model/train_config.json \
#   --resume=true

[自动备份] 已同步到新的 Drive 目录: 2026-04-06 06:01:29
INFO 2026-04-06 06:01:36 ot_train.py:197 {'batch_size': 8,
 'checkpoint_path': '/content/lerobot/outputs/train/pusht_act_resume_20260406_060036/checkpoints/last',
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.8,
                           

In [ ]:
# #重新开始训练
# !cd /content/lerobot && python src/lerobot/scripts/lerobot_train.py \
#   --dataset.repo_id=lerobot/pusht \
#   --policy.type=act \
#   --output_dir=outputs/train/pusht_act \
#   --job_name=pusht_act_training \
#   --policy.device=cuda \
#   --wandb.enable=False \
#   --policy.repo_id=${LDD11}/pusht_act_policy

INFO 2026-04-05 10:42:13 ot_train.py:197 {'batch_size': 8,
 'checkpoint_path': None,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.8,
                                                                                   1.2]},
                                                         't

In [ ]:
#训练因为目录冲突停止，继续训练
# 1) 进入仓库目录
%cd /content/lerobot

# 2) 确保旧训练目录存在；如果 Colab 断开过，就从 Drive 恢复
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

OLD_LOCAL_DIR = "/content/lerobot/outputs/train/pusht_act"
OLD_DRIVE_DIR = "/content/drive/MyDrive/pusht_act_backup"

if not os.path.exists(OLD_LOCAL_DIR):
    shutil.copytree(OLD_DRIVE_DIR, OLD_LOCAL_DIR)

# 3) 删除有冲突的 last
!rm -rf /content/lerobot/outputs/train/pusht_act/checkpoints/last

# 4) 从最新真实 checkpoint 继续，而不是从 last 继续
#更改保存频率
!python src/lerobot/scripts/lerobot_train.py \
  --config_path=/content/lerobot/outputs/train/pusht_act/checkpoints/080000/pretrained_model/train_config.json \
  --resume=true \
  --output_dir=outputs/train/pusht_act \
  --job_name=pusht_act_training \
  --save_freq=10000 \
  --num_workers=2


/content/lerobot
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[自动备份] 已同步到 Drive: 2026-04-08 02:44:37
INFO 2026-04-08 02:44:49 ot_train.py:197 {'batch_size': 8,
 'checkpoint_path': '/content/lerobot/outputs/train/pusht_act/checkpoints/080000',
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1

In [ ]:

!cp -r /content/lerobot/outputs/train/pusht_act/checkpoints/100000 /content/drive/MyDrive/lerobot_backup/

In [ ]:
import os
import shutil

LOCAL_OUTPUT_DIR = "/content/lerobot/outputs/train/pusht_act"
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/pusht_act_backup"

if os.path.exists(DRIVE_BACKUP_DIR):
    shutil.rmtree(DRIVE_BACKUP_DIR)
shutil.copytree(LOCAL_OUTPUT_DIR, DRIVE_BACKUP_DIR)

print("最终备份完成")

[自动备份失败] [('/content/lerobot/outputs/train/pusht_act/checkpoints/100000/pretrained_model/model.safetensors', '/content/drive/MyDrive/pusht_act_backup_tmp/checkpoints/100000/pretrained_model/model.safetensors', '[Errno 2] No such file or directory'), ('/content/lerobot/outputs/train/pusht_act/checkpoints/100000/pretrained_model/policy_preprocessor.json', '/content/drive/MyDrive/pusht_act_backup_tmp/checkpoints/100000/pretrained_model/policy_preprocessor.json', "[Errno 2] No such file or directory: '/content/drive/MyDrive/pusht_act_backup_tmp/checkpoints/100000/pretrained_model/policy_preprocessor.json'"), ('/content/lerobot/outputs/train/pusht_act/checkpoints/100000/pretrained_model/policy_preprocessor_step_3_normalizer_processor.safetensors', '/content/drive/MyDrive/pusht_act_backup_tmp/checkpoints/100000/pretrained_model/policy_preprocessor_step_3_normalizer_processor.safetensors', "[Errno 2] No such file or directory: '/content/drive/MyDrive/pusht_act_backup_tmp/checkpoints/100000/pr

In [ ]:
#清除旧登录状态
import os
import shutil

# 1. 删除 huggingface 本地缓存的 token
paths_to_remove = [
    os.path.expanduser("~/.cache/huggingface"),
    os.path.expanduser("~/.huggingface"),
]

for p in paths_to_remove:
    if os.path.exists(p):
        shutil.rmtree(p)
        print("已删除：", p)
    else:
        print("不存在：", p)

# 2. 清空当前环境变量里可能残留的 token
for k in ["HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HUGGINGFACEHUB_API_TOKEN"]:
    if k in os.environ:
        del os.environ[k]
        print("已清除环境变量：", k)

print("旧登录状态已清理")

已删除： /root/.cache/huggingface
不存在： /root/.huggingface
旧登录状态已清理


In [5]:
!pip install -U huggingface_hub

from huggingface_hub import login, HfApi

# 1. 重新登录
login()   # 这里粘贴新的 Hugging Face token

# 2. 初始化 API
api = HfApi()

# 3. 看当前登录信息
me = api.whoami()
print("当前登录信息：")
print(me)

# 4. 创建仓库
repo_id = "LDD11/pusht-act-model"
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
print("仓库已创建或已存在：", repo_id)

# 5. 上传文件夹
model_dir = "/content/lerobot/outputs/train/pusht_act/checkpoints/100000/pretrained_model"
api.upload_folder(
    folder_path=model_dir,
    repo_id=repo_id,
    repo_type="model",
)
print("上传完成")

# 6. 查看仓库文件
files = api.list_repo_files(repo_id=repo_id, repo_type="model")
print("仓库文件：")
for f in files:
    print(f)

当前登录信息：
{'type': 'user', 'id': '69d20db311b71ec1b52c8d87', 'name': 'LDD11', 'fullname': 'DDL', 'email': 'kpmglyd@icloud.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1777593600, 'isPro': False, 'avatarUrl': '/avatars/77f5b2696d822cc1bfafd24e76bb4fd5.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'LDD15', 'role': 'write', 'createdAt': '2026-04-09T16:54:43.919Z'}}}
仓库已创建或已存在： LDD11/pusht-act-model


ValueError: Provided path: '/content/lerobot/outputs/train/pusht_act/checkpoints/100000/pretrained_model' is not a directory

In [7]:
#上传文件验证
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files("LDD11/pusht-act-model", repo_type="model")
for f in files:
    print(f)

.gitattributes
config.json
model.safetensors
policy_postprocessor.json
policy_postprocessor_step_0_unnormalizer_processor.safetensors
policy_preprocessor.json
policy_preprocessor_step_3_normalizer_processor.safetensors
train_config.json


In [6]:
#从HuggingFace上拉取文件
from huggingface_hub import snapshot_download

local_dir = snapshot_download(
    repo_id="LDD11/pusht-act-model",
    repo_type="model"
)
print(local_dir)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

/root/.cache/huggingface/hub/models--LDD11--pusht-act-model/snapshots/1e12fc1eb4f3f477f05b46e806155848ed364b96


In [8]:
#HuggingFace Debug
# =========================
# 0. 安装依赖
# =========================
!pip install -U huggingface_hub

# =========================
# 1. 挂载 Drive（如果模型可能在 Drive 备份里）
# =========================
from google.colab import drive
drive.mount('/content/drive')

# =========================
# 2. 设置用户名和仓库名
#    这里一定要写你真正的 HF 用户名
# =========================
HF_USERNAME = "LDD11"   # 改成你 Hugging Face 首页右上角看到的真实用户名
HF_REPO_NAME = "pusht-act-model"
REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"

# =========================
# 3. 自动查找模型目录
# =========================
import os

candidate_dirs = [
    "/content/lerobot/outputs/train/pusht_act/checkpoints/100000/pretrained_model",
    "/content/drive/MyDrive/lerobot_backup/100000/pretrained_model",
    "/content/drive/MyDrive/pusht_act_backup/checkpoints/100000/pretrained_model",
    "/content/lerobot/outputs/train/pusht_act/checkpoints/last/pretrained_model",
    "/content/drive/MyDrive/pusht_act_backup/checkpoints/last/pretrained_model",
]

MODEL_DIR = None
for path in candidate_dirs:
    if os.path.exists(path):
        MODEL_DIR = path
        break

if MODEL_DIR is None:
    raise FileNotFoundError("没找到可上传的 pretrained_model 目录，请先检查 checkpoint 文件。")

print("找到模型目录：", MODEL_DIR)

# =========================
# 4. 登录 Hugging Face
#    注意：这里输入的是 Hugging Face token，不是 wandb token
# =========================
from huggingface_hub import login, HfApi
login()

# =========================
# 5. 检查当前登录账号是谁
# =========================
api = HfApi()
me = api.whoami()
print("当前登录账号信息：")
print(me)

# =========================
# 6. 显式创建 repo
#    exist_ok=True 表示如果已存在就直接继续
# =========================
api.create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    private=False,     # 你也可以改成 True
    exist_ok=True
)
print("仓库已确认存在：", REPO_ID)

# =========================
# 7. 先试着列出仓库文件，确认你有权限访问
# =========================
files_before = api.list_repo_files(repo_id=REPO_ID, repo_type="model")
print("上传前仓库文件：", files_before)

# =========================
# 8. 上传整个文件夹
# =========================
api.upload_folder(
    folder_path=MODEL_DIR,
    repo_id=REPO_ID,
    repo_type="model",
)
print("上传完成。")

# =========================
# 9. 再次检查仓库内容
# =========================
files_after = api.list_repo_files(repo_id=REPO_ID, repo_type="model")
print("上传后仓库文件：")
for f in files_after:
    print(f)

print("仓库地址：")
print(f"https://huggingface.co/{REPO_ID}")

MessageError: Error: credential propagation was unsuccessful

In [ ]:
#评估-本地目录
import os

candidate_dirs = [
    "/content/lerobot/outputs/train/pusht_act/checkpoints/100000/pretrained_model",
    "/content/lerobot/outputs/train/pusht_act/checkpoints/last/pretrained_model",
    "/content/drive/MyDrive/lerobot_backup/100000/pretrained_model",
    "/content/drive/MyDrive/pusht_act_backup/checkpoints/100000/pretrained_model",
]

model_dir = None
for p in candidate_dirs:
    if os.path.exists(p):
        model_dir = p
        break

print("model_dir =", model_dir)

model_dir = /content/lerobot/outputs/train/pusht_act/checkpoints/100000/pretrained_model


In [ ]:
#评估-50局
!lerobot-eval \
  --policy.path="{model_dir}" \
  --env.type=pusht \
  --eval.n_episodes=50

WARNING 2026-04-08 04:21:10 figs/eval.py:65 No job name provided, using 'pusht_act' as job name.
INFO 2026-04-08 04:21:10 bot_eval.py:507 {'env': {'disable_env_checker': True,
         'episode_length': 300,
         'features': {'action': {'shape': (2,),
                                 'type': <FeatureType.ACTION: 'ACTION'>},
                      'agent_pos': {'shape': (2,),
                                    'type': <FeatureType.STATE: 'STATE'>},
                      'pixels': {'shape': (384, 384, 3),
                                 'type': <FeatureType.VISUAL: 'VISUAL'>}},
         'features_map': {'action': 'action',
                          'agent_pos': 'observation.state',
                          'environment_state': 'observation.environment_state',
                          'pixels': 'observation.image'},
         'fps': 10,
         'max_parallel_tasks': 1,
         'obs_type': 'pixels_agent_pos',
         'observation_height': 384,
         'observation_width': 384,
  

In [ ]:
!lerobot-eval \
  --policy.path="/content/lerobot/outputs/train/pusht_act/checkpoints/020000/pretrained_model" \
  --env.type=pusht \
  --eval.batch_size=50 \
  --eval.n_episodes=50 \
  --policy.device=cuda

WARNING 2026-04-08 13:28:23 figs/eval.py:65 No job name provided, using 'pusht_act' as job name.
INFO 2026-04-08 13:28:23 bot_eval.py:507 {'env': {'disable_env_checker': True,
         'episode_length': 300,
         'features': {'action': {'shape': (2,),
                                 'type': <FeatureType.ACTION: 'ACTION'>},
                      'agent_pos': {'shape': (2,),
                                    'type': <FeatureType.STATE: 'STATE'>},
                      'pixels': {'shape': (384, 384, 3),
                                 'type': <FeatureType.VISUAL: 'VISUAL'>}},
         'features_map': {'action': 'action',
                          'agent_pos': 'observation.state',
                          'environment_state': 'observation.environment_state',
                          'pixels': 'observation.image'},
         'fps': 10,
         'max_parallel_tasks': 1,
         'obs_type': 'pixels_agent_pos',
         'observation_height': 384,
         'observation_width': 384,
  

第一轮训练结束，结果正确率很低，第二轮训练修改chunk_size参数

自动备份线程一直运行，但是备份的目录是原始目录，不是调参后目录

In [ ]:
# 第二轮训练，chunk_size=50
CHUNK_SIZE=50
RUN_NAME="pusht_act_chunk50_quick"

!cd /content/lerobot && python src/lerobot/scripts/lerobot_train.py \
  --dataset.repo_id=lerobot/pusht \
  --policy.type=act \
  --output_dir=chunk50_runs/pusht_act_chunk50_quick_40000 \
  --job_name=pusht_act_chunk50_quick_40000 \
  --policy.device=cuda \
  --wandb.enable=False \
  --steps=40000 \
  --policy.chunk_size=50 \
  --policy.n_action_steps=50 \
  --policy.repo_id=LDD11/pusht_act_chunk50_quick_40000

INFO 2026-04-08 14:29:50 ot_train.py:197 {'batch_size': 8,
 'checkpoint_path': None,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.8,
                                                                                   1.2]},
                                                         't

第二轮测试到40000步时，成功率 10局10%吗，50局6%
继续训练到100000步，成功率：10局10%
50局6%，但是完成任务的时间显著缩短了

In [ ]:
#测试chunk_size=50
!ls -R /content/lerobot/chunk50_runs/pusht_act_chunk50_quick_40000/checkpoints | head -100
!lerobot-eval \
  --policy.path="/content/lerobot/chunk50_runs/pusht_act_chunk50_quick_40000/checkpoints/last/pretrained_model" \
  --env.type=pusht \
  --eval.batch_size=50 \
  --eval.n_episodes=50 \
  --policy.device=cuda

/content/lerobot/chunk50_runs/pusht_act_chunk50_quick_40000/checkpoints:
020000
040000
060000
080000
100000
last

/content/lerobot/chunk50_runs/pusht_act_chunk50_quick_40000/checkpoints/020000:
pretrained_model
training_state

/content/lerobot/chunk50_runs/pusht_act_chunk50_quick_40000/checkpoints/020000/pretrained_model:
config.json
model.safetensors
policy_postprocessor.json
policy_postprocessor_step_0_unnormalizer_processor.safetensors
policy_preprocessor.json
policy_preprocessor_step_3_normalizer_processor.safetensors
train_config.json

/content/lerobot/chunk50_runs/pusht_act_chunk50_quick_40000/checkpoints/020000/training_state:
optimizer_param_groups.json
optimizer_state.safetensors
rng_state.safetensors
training_step.json

/content/lerobot/chunk50_runs/pusht_act_chunk50_quick_40000/checkpoints/040000:
pretrained_model
training_state

/content/lerobot/chunk50_runs/pusht_act_chunk50_quick_40000/checkpoints/040000/pretrained_model:
config.json
model.safetensors
policy_postprocessor

In [ ]:
#继续训练到100000
!cd /content/lerobot && python src/lerobot/scripts/lerobot_train.py \
  --config_path=/content/lerobot/chunk50_runs/pusht_act_chunk50_quick_40000/checkpoints/last/pretrained_model/train_config.json \
  --resume=true \
  --steps=100000

INFO 2026-04-08 15:21:06 ot_train.py:197 {'batch_size': 8,
 'checkpoint_path': '/content/lerobot/chunk50_runs/pusht_act_chunk50_quick_40000/checkpoints/last',
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.8,
                                                                           

第三路训练chunksize50 n_action 20
成功率显著提升，50局24% 12/50
继续训练到80000步后，成功率不变，行为质量提高，选为基准模型
action10，成功率下降

In [ ]:
#第三轮训练 chunksize50 n_action 20
#第四轮训练 chunksize50 n_action 15 #针对n_action_steps调优
#chunksize50 n_action 10 #针对n_action_steps调优
!cd /content/lerobot && python src/lerobot/scripts/lerobot_train.py \
  --dataset.repo_id=lerobot/pusht \
  --policy.type=act \
  --output_dir=chunk50_runs/pusht_act_chunk50_action10_40000 \
  --job_name=pusht_act_chunk50_action10_40000 \
  --policy.device=cuda \
  --wandb.enable=False \
  --steps=40000 \
  --policy.chunk_size=50 \
  --policy.n_action_steps=10 \
  --policy.repo_id=LDD11/pusht_act_chunk50_action10_40000

INFO 2026-04-09 08:03:04 ot_train.py:197 {'batch_size': 8,
 'checkpoint_path': None,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.8,
                                                                                   1.2]},
                                                         't

In [ ]:
#继续训练到80000步
!cd /content/lerobot && python src/lerobot/scripts/lerobot_train.py \
  --config_path=/content/lerobot/chunk50_runs/pusht_act_chunk50_action20_40000/checkpoints/last/pretrained_model/train_config.json \
  --resume=true \
  --steps=80000

INFO 2026-04-09 06:29:29 ot_train.py:197 {'batch_size': 8,
 'checkpoint_path': '/content/lerobot/chunk50_runs/pusht_act_chunk50_action20_40000/checkpoints/last',
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.8,
                                                                        

In [ ]:
#测试-chunksize50 n_action 10
!lerobot-eval \
  --policy.path="/content/lerobot/chunk50_runs/pusht_act_chunk50_action10_40000/checkpoints/last/pretrained_model" \
  --env.type=pusht \
  --eval.batch_size=50 \
  --eval.n_episodes=50 \
  --policy.device=cuda

WARNING 2026-04-09 08:38:39 figs/eval.py:65 No job name provided, using 'pusht_act' as job name.
INFO 2026-04-09 08:38:39 bot_eval.py:507 {'env': {'disable_env_checker': True,
         'episode_length': 300,
         'features': {'action': {'shape': (2,),
                                 'type': <FeatureType.ACTION: 'ACTION'>},
                      'agent_pos': {'shape': (2,),
                                    'type': <FeatureType.STATE: 'STATE'>},
                      'pixels': {'shape': (384, 384, 3),
                                 'type': <FeatureType.VISUAL: 'VISUAL'>}},
         'features_map': {'action': 'action',
                          'agent_pos': 'observation.state',
                          'environment_state': 'observation.environment_state',
                          'pixels': 'observation.image'},
         'fps': 10,
         'max_parallel_tasks': 1,
         'obs_type': 'pixels_agent_pos',
         'observation_height': 384,
         'observation_width': 384,
  

In [ ]:
!cp -r /content/drive/MyDrive/lerobot_backup_2026_04_09_chunk50_action20_40000/checkpoints \
      /content/lerobot/chunk50_runs/pusht_act_chunk50_action20_40000/

In [ ]:
#上传本次实验结果到huggingface
from huggingface_hub import create_repo

repo_id = "LDD11/pusht_act_chunk50_quick_40000_backup"

create_repo(
    repo_id=repo_id,
    repo_type="model",
    exist_ok=True
)

print("仓库已就绪：", repo_id)

仓库已就绪： LDD11/pusht_act_chunk50_quick_40000_backup


In [ ]:
#上传
from huggingface_hub import HfApi

api = HfApi()
repo_id = "LDD11/pusht_act_chunk50_quick_40000_backup"
local_folder = "/content/lerobot/chunk50_runs/pusht_act_chunk50_quick_40000"

api.upload_folder(
    folder_path=local_folder,
    repo_id=repo_id,
    repo_type="model",
    commit_message="Backup full training results: checkpoints 020000-100000 and last"
)

print("上传完成")

[自动备份] 已同步到 Drive: 2026-04-08 16:30:36


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...zer_processor.safetensors: 100%|##########| 4.20kB / 4.20kB            

  ...zer_processor.safetensors: 100%|##########| 4.20kB / 4.20kB            

...

  ...zer_processor.safetensors: 100%|##########| 4.20kB / 4.20kB            

  ...zer_processor.safetensors: 100%|##########| 4.20kB / 4.20kB            

  ...d_model/model.safetensors:  58%|#####8    |  120MB /  206MB            

  ...d_model/model.safetensors:   0%|          |  627kB /  206MB            

上传完成


In [ ]:
#上传结果到Google Drive
backup_dir = "/content/drive/MyDrive/lerobot_backup_2026_04_09_chunk50_action20_40000"

!mkdir -p "{backup_dir}"

!cp -r /content/lerobot/chunk50_runs/pusht_act_chunk50_action20_40000/checkpoints \
      "{backup_dir}/"

!ls -R "{backup_dir}" | head -100

/content/drive/MyDrive/lerobot_backup_2026_04_09_chunk50_action20_40000:
checkpoints

/content/drive/MyDrive/lerobot_backup_2026_04_09_chunk50_action20_40000/checkpoints:
020000
040000
060000
080000
last

/content/drive/MyDrive/lerobot_backup_2026_04_09_chunk50_action20_40000/checkpoints/020000:
pretrained_model
training_state

/content/drive/MyDrive/lerobot_backup_2026_04_09_chunk50_action20_40000/checkpoints/020000/pretrained_model:
config.json
model.safetensors
policy_postprocessor.json
policy_postprocessor_step_0_unnormalizer_processor.safetensors
policy_preprocessor.json
policy_preprocessor_step_3_normalizer_processor.safetensors
train_config.json

/content/drive/MyDrive/lerobot_backup_2026_04_09_chunk50_action20_40000/checkpoints/020000/training_state:
optimizer_param_groups.json
optimizer_state.safetensors
rng_state.safetensors
training_step.json

/content/drive/MyDrive/lerobot_backup_2026_04_09_chunk50_action20_40000/checkpoints/040000:
pretrained_model
training_state

/content

In [ ]:
!ls /content/lerobot/chunk50_runs
!ls -R /content/lerobot/chunk50_runs/pusht_act_chunk50_action20_40000 | head -100

pusht_act_chunk50_action10_40000  pusht_act_chunk50_action20_40000
/content/lerobot/chunk50_runs/pusht_act_chunk50_action20_40000:
020000
040000
060000
080000
last

/content/lerobot/chunk50_runs/pusht_act_chunk50_action20_40000/020000:
pretrained_model
training_state

/content/lerobot/chunk50_runs/pusht_act_chunk50_action20_40000/020000/pretrained_model:
config.json
model.safetensors
policy_postprocessor.json
policy_postprocessor_step_0_unnormalizer_processor.safetensors
policy_preprocessor.json
policy_preprocessor_step_3_normalizer_processor.safetensors
train_config.json

/content/lerobot/chunk50_runs/pusht_act_chunk50_action20_40000/020000/training_state:
optimizer_param_groups.json
optimizer_state.safetensors
rng_state.safetensors
training_step.json

/content/lerobot/chunk50_runs/pusht_act_chunk50_action20_40000/040000:
pretrained_model
training_state

/content/lerobot/chunk50_runs/pusht_act_chunk50_action20_40000/040000/pretrained_model:
config.json
model.safetensors
policy_postpro

In [ ]:
#第四轮训练，打开图像增强
!cd /content/lerobot && python src/lerobot/scripts/lerobot_train.py \
  --dataset.repo_id=lerobot/pusht \
  --policy.type=act \
  --output_dir=chunk50_runs/pusht_act_chunk50_action20_aug_40000 \
  --job_name=pusht_act_chunk50_action20_aug_40000 \
  --policy.device=cuda \
  --wandb.enable=False \
  --steps=40000 \
  --policy.chunk_size=50 \
  --policy.n_action_steps=20 \
  --dataset.image_transforms.enable=true \
  --policy.repo_id=LDD11/pusht_act_chunk50_action20_aug_40000

INFO 2026-04-09 08:57:51 ot_train.py:197 {'batch_size': 8,
 'checkpoint_path': None,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': True,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.8,
                                                                                   1.2]},
                                                         'ty

In [ ]:
#测试-chunksize50 n_action 10
#测试-chunksize50 n_action 20 图像增强
!lerobot-eval \
  --policy.path="/content/lerobot/chunk50_runs/pusht_act_chunk50_action20_aug_40000/checkpoints/040000/pretrained_model" \
  --env.type=pusht \
  --eval.batch_size=50 \
  --eval.n_episodes=50 \
  --policy.device=cuda

WARNING 2026-04-09 09:37:41 figs/eval.py:65 No job name provided, using 'pusht_act' as job name.
INFO 2026-04-09 09:37:41 bot_eval.py:507 {'env': {'disable_env_checker': True,
         'episode_length': 300,
         'features': {'action': {'shape': (2,),
                                 'type': <FeatureType.ACTION: 'ACTION'>},
                      'agent_pos': {'shape': (2,),
                                    'type': <FeatureType.STATE: 'STATE'>},
                      'pixels': {'shape': (384, 384, 3),
                                 'type': <FeatureType.VISUAL: 'VISUAL'>}},
         'features_map': {'action': 'action',
                          'agent_pos': 'observation.state',
                          'environment_state': 'observation.environment_state',
                          'pixels': 'observation.image'},
         'fps': 10,
         'max_parallel_tasks': 1,
         'obs_type': 'pixels_agent_pos',
         'observation_height': 384,
         'observation_width': 384,
  

In [ ]:
#第五轮训练，chunk=50，action改为1，准备加入temporal ensembling

!cd /content/lerobot && python src/lerobot/scripts/lerobot_train.py \
    --dataset.repo_id=lerobot/pusht \
    --policy.type=act \
    --output_dir=chunk50_runs/pusht_act_chunk50_action1_60000 \
    --job_name=pusht_act_chunk50_action1_60000 \
    --policy.device=cuda \
    --wandb.enable=False \
    --steps=60000 \
    --policy.chunk_size=50 \
    --policy.n_action_steps=1 \
    --policy.repo_id=LDD11/pusht_act_chunk50_action1_60000

INFO 2026-04-09 09:52:49 ot_train.py:197 {'batch_size': 8,
 'checkpoint_path': None,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.8,
                                                                                   1.2]},
                                                         't

In [ ]:
#测试Temporal Ensembling
!lerobot-eval \
  --policy.path="/content/lerobot/chunk50_runs/pusht_act_chunk50_action1_60000/checkpoints/last/pretrained_model" \
  --env.type=pusht \
  --eval.batch_size=50 \
  --eval.n_episodes=50 \
  --policy.device=cuda \
  --policy.temporal_ensemble_coeff=0.005

WARNING 2026-04-09 10:57:58 figs/eval.py:65 No job name provided, using 'pusht_act' as job name.
INFO 2026-04-09 10:57:58 bot_eval.py:507 {'env': {'disable_env_checker': True,
         'episode_length': 300,
         'features': {'action': {'shape': (2,),
                                 'type': <FeatureType.ACTION: 'ACTION'>},
                      'agent_pos': {'shape': (2,),
                                    'type': <FeatureType.STATE: 'STATE'>},
                      'pixels': {'shape': (384, 384, 3),
                                 'type': <FeatureType.VISUAL: 'VISUAL'>}},
         'features_map': {'action': 'action',
                          'agent_pos': 'observation.state',
                          'environment_state': 'observation.environment_state',
                          'pixels': 'observation.image'},
         'fps': 10,
         'max_parallel_tasks': 1,
         'obs_type': 'pixels_agent_pos',
         'observation_height': 384,
         'observation_width': 384,
  

In [ ]:
#开始Diffusion Policy路线
!cd /content/lerobot && python src/lerobot/scripts/lerobot_train.py \
  --dataset.repo_id=lerobot/pusht \
  --policy.type=diffusion \
  --output_dir=diffusion_runs/pusht_dp_base_40000 \
  --job_name=pusht_dp_base_40000 \
  --policy.device=cuda \
  --wandb.enable=False \
  --steps=40000 \
  --policy.repo_id=LDD11/pusht_dp_base_40000

INFO 2026-04-09 11:03:36 ot_train.py:197 {'batch_size': 8,
 'checkpoint_path': None,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.8,
                                                                                   1.2]},
                                                         't

In [ ]:
!cd /content/lerobot && python src/lerobot/scripts/lerobot_train.py \
  --config_path=/content/lerobot/diffusion_runs/pusht_dp_base_40000/checkpoints/last/pretrained_model/train_config.json \
  --resume=true \
  --steps=80000

In [ ]:
#diffusion测试
#减少num_inference_steps
!lerobot-eval \
  --policy.path="/content/lerobot/diffusion_runs/pusht_dp_base_40000/checkpoints/last/pretrained_model" \
  --env.type=pusht \
  --eval.batch_size=50 \
  --eval.n_episodes=50 \
  --policy.device=cuda \
  --policy.num_inference_steps=150

WARNING 2026-04-09 12:27:10 figs/eval.py:65 No job name provided, using 'pusht_diffusion' as job name.
INFO 2026-04-09 12:27:10 bot_eval.py:507 {'env': {'disable_env_checker': True,
         'episode_length': 300,
         'features': {'action': {'shape': (2,),
                                 'type': <FeatureType.ACTION: 'ACTION'>},
                      'agent_pos': {'shape': (2,),
                                    'type': <FeatureType.STATE: 'STATE'>},
                      'pixels': {'shape': (384, 384, 3),
                                 'type': <FeatureType.VISUAL: 'VISUAL'>}},
         'features_map': {'action': 'action',
                          'agent_pos': 'observation.state',
                          'environment_state': 'observation.environment_state',
                          'pixels': 'observation.image'},
         'fps': 10,
         'max_parallel_tasks': 1,
         'obs_type': 'pixels_agent_pos',
         'observation_height': 384,
         'observation_width': 3

In [ ]:
#继续测试，step下调到18，学习率2e-5
!cd /content/lerobot && python src/lerobot/scripts/lerobot_train.py \
  --dataset.repo_id=lerobot/pusht \
  --policy.type=act \
  --output_dir=chunk50_runs/pusht_act_chunk50_action18_lr2e5_40000 \
  --job_name=pusht_act_chunk50_action18_lr2e5_40000 \
  --policy.device=cuda \
  --wandb.enable=False \
  --steps=40000 \
  --policy.chunk_size=50 \
  --policy.n_action_steps=18 \
  --optimizer.lr=2e-5 \
  --policy.optimizer_lr=2e-5 \
  --policy.optimizer_lr_backbone=2e-5 \
  --policy.repo_id=LDD11/pusht_act_chunk50_action18_lr2e5_40000

INFO 2026-04-09 12:42:38 ot_train.py:197 {'batch_size': 8,
 'checkpoint_path': None,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.8,
                                                                                   1.2]},
                                                         't

In [ ]:
#测试结果不错，继续训练，从40000到70000，每5000记录一次
!cd /content/lerobot && python src/lerobot/scripts/lerobot_train.py \
  --config_path=/content/lerobot/chunk50_runs/pusht_act_chunk50_action18_lr2e5_40000/checkpoints/040000/pretrained_model/train_config.json \
  --resume=true \
  --steps=70000 \
  --save_freq=5000 \
  --output_dir=chunk50_runs/pusht_act_chunk50_action18_lr2e5_from040000_to70000_save5000 \
  --job_name=pusht_act_chunk50_action18_lr2e5_from040000_to70000_save5000

INFO 2026-04-09 13:26:56 ot_train.py:197 {'batch_size': 8,
 'checkpoint_path': '/content/lerobot/chunk50_runs/pusht_act_chunk50_action18_lr2e5_40000/checkpoints/040000',
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.8,
                                                                

In [ ]:
#测试-step下调到18，学习率2e-5
!lerobot-eval \
  --policy.path="/content/lerobot/chunk50_runs/pusht_act_chunk50_action18_lr2e5_40000/checkpoints/last/pretrained_model" \
  --env.type=pusht \
  --eval.batch_size=50 \
  --eval.n_episodes=50 \
  --policy.device=cuda

WARNING 2026-04-09 13:24:07 figs/eval.py:65 No job name provided, using 'pusht_act' as job name.
INFO 2026-04-09 13:24:07 bot_eval.py:507 {'env': {'disable_env_checker': True,
         'episode_length': 300,
         'features': {'action': {'shape': (2,),
                                 'type': <FeatureType.ACTION: 'ACTION'>},
                      'agent_pos': {'shape': (2,),
                                    'type': <FeatureType.STATE: 'STATE'>},
                      'pixels': {'shape': (384, 384, 3),
                                 'type': <FeatureType.VISUAL: 'VISUAL'>}},
         'features_map': {'action': 'action',
                          'agent_pos': 'observation.state',
                          'environment_state': 'observation.environment_state',
                          'pixels': 'observation.image'},
         'fps': 10,
         'max_parallel_tasks': 1,
         'obs_type': 'pixels_agent_pos',
         'observation_height': 384,
         'observation_width': 384,
  

In [ ]:
# 批量测试，只保留最佳
# 从 40000 到 70000，每 5000 测一次
# 每个 checkpoint 测三次并取平均
# 只保留最佳结果的视频组
import os
import re
import json
import shutil
import subprocess

# ========= 配置区 =========
steps = list(range(40000, 70001, 5000))
base_dir = "/content/lerobot/chunk50_runs/pusht_act_chunk50_action18_lr2e5_from040000_to70000_save5000/checkpoints"
log_dir = "/content/lerobot/eval_logs_best_video_verbose_action18"
best_video_root = "/content/lerobot/best_eval_videos_verbose_action18"

os.makedirs(log_dir, exist_ok=True)
os.makedirs(best_video_root, exist_ok=True)

def parse_metrics(text: str):
    pc_success = None
    avg_sum_reward = None
    avg_max_reward = None
    output_dir = None

    m_out = re.search(r"Output dir:\s*(.+)", text)
    if m_out:
        output_dir = m_out.group(1).strip()

    m_sum = re.search(r"'avg_sum_reward':\s*([0-9eE\.\-]+)", text)
    m_max = re.search(r"'avg_max_reward':\s*([0-9eE\.\-]+)", text)
    m_suc = re.search(r"'pc_success':\s*([0-9eE\.\-]+)", text)

    if m_sum:
        avg_sum_reward = float(m_sum.group(1))
    if m_max:
        avg_max_reward = float(m_max.group(1))
    if m_suc:
        pc_success = float(m_suc.group(1))

    return {
        "pc_success": pc_success,
        "avg_sum_reward": avg_sum_reward,
        "avg_max_reward": avg_max_reward,
        "output_dir": output_dir,
    }

def better(a, b):
    if b is None:
        return True
    ka = (a["avg_pc_success"], a["avg_sum_reward"], a["avg_max_reward"])
    kb = (b["avg_pc_success"], b["avg_sum_reward"], b["avg_max_reward"])
    return ka > kb

def better_single_run(a, b):
    if b is None:
        return True
    ka = (a["pc_success"], a["avg_sum_reward"], a["avg_max_reward"])
    kb = (b["pc_success"], b["avg_sum_reward"], b["avg_max_reward"])
    return ka > kb

summary = []
global_best = None
total_ckpts = len(steps)

for idx, step in enumerate(steps, start=1):
    step_str = f"{step:06d}"
    model_path = os.path.join(base_dir, step_str, "pretrained_model")
    step_log_path = os.path.join(log_dir, f"eval_{step_str}_summary.txt")

    print("\n" + "=" * 80)
    print(f"[{idx}/{total_ckpts}] 开始评测 checkpoint: {step_str}")
    print("=" * 80)

    if not os.path.isdir(model_path):
        msg = f"Checkpoint {step_str} not found, skipped."
        print(msg)
        with open(step_log_path, "w", encoding="utf-8") as f:
            f.write(msg + "\n")
        continue

    runs = []
    best_run = None

    # ===== 每个 checkpoint 测三次 =====
    for run_idx in range(1, 4):
        print(f"\n--- Checkpoint {step_str} | Run {run_idx}/3 开始 ---")

        cmd = [
            "lerobot-eval",
            f"--policy.path={model_path}",
            "--env.type=pusht",
            "--eval.batch_size=50",
            "--eval.n_episodes=50",
            "--policy.device=cuda",
        ]

        result = subprocess.run(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True
        )

        metrics = parse_metrics(result.stdout)
        metrics["run_idx"] = run_idx
        metrics["raw_output"] = result.stdout

        runs.append(metrics)

        print(f"Run {run_idx}/3 完成：")
        print(f"  pc_success     = {metrics['pc_success']}")
        print(f"  avg_sum_reward = {metrics['avg_sum_reward']}")
        print(f"  avg_max_reward = {metrics['avg_max_reward']}")
        print(f"  output_dir     = {metrics['output_dir']}")

        if better_single_run(metrics, best_run):
            best_run = metrics
            print(f"  >>> 当前 checkpoint {step_str} 内部最佳 run 已更新为 Run {run_idx}")

    # ===== 计算三次平均 =====
    avg_pc_success = sum(r["pc_success"] for r in runs) / len(runs)
    avg_sum_reward = sum(r["avg_sum_reward"] for r in runs) / len(runs)
    avg_max_reward = sum(r["avg_max_reward"] for r in runs) / len(runs)

    checkpoint_result = {
        "checkpoint": step_str,
        "avg_pc_success": avg_pc_success,
        "avg_sum_reward": avg_sum_reward,
        "avg_max_reward": avg_max_reward,
        "best_run_idx": best_run["run_idx"],
        "best_run_pc_success": best_run["pc_success"],
        "best_run_avg_sum_reward": best_run["avg_sum_reward"],
        "best_run_avg_max_reward": best_run["avg_max_reward"],
        "kept_video_dir": None,
    }

    print("\n" + "-" * 80)
    print(f"Checkpoint {step_str} 三次平均结果：")
    print(f"  avg_pc_success = {avg_pc_success:.4f}")
    print(f"  avg_sum_reward = {avg_sum_reward:.4f}")
    print(f"  avg_max_reward = {avg_max_reward:.4f}")
    print(f"  最佳单次 run   = Run {best_run['run_idx']}")
    print("-" * 80)

    # ===== 只保留该 checkpoint 内部最好一次的视频 =====
    kept_video_dir = None
    for r in runs:
        out_dir = r["output_dir"]
        if not out_dir:
            continue

        src_video_dir = os.path.join(out_dir, "videos")

        if r["run_idx"] == best_run["run_idx"]:
            dst_dir = os.path.join(best_video_root, f"checkpoint_{step_str}")
            if os.path.exists(dst_dir):
                shutil.rmtree(dst_dir)
            if os.path.isdir(src_video_dir):
                shutil.copytree(src_video_dir, dst_dir)
                kept_video_dir = dst_dir
                print(f"已保留最佳视频：{dst_dir}")
        else:
            if os.path.isdir(out_dir):
                shutil.rmtree(out_dir, ignore_errors=True)
                print(f"已删除非最佳 run 的评测目录：{out_dir}")

    if best_run["output_dir"] and os.path.isdir(best_run["output_dir"]):
        shutil.rmtree(best_run["output_dir"], ignore_errors=True)
        print("已删除最佳 run 原始评测目录，仅保留复制后的视频目录。")

    checkpoint_result["kept_video_dir"] = kept_video_dir

    # ===== 判断是否刷新全局最佳 =====
    if better(checkpoint_result, global_best):
        global_best = checkpoint_result
        print("\n" + "🔥" * 25)
        print(f">>> NEW BEST CHECKPOINT FOUND: {step_str}")
        print(f">>> avg_pc_success = {avg_pc_success:.4f}")
        print(f">>> avg_sum_reward = {avg_sum_reward:.4f}")
        print(f">>> avg_max_reward = {avg_max_reward:.4f}")
        print("🔥" * 25 + "\n")
    else:
        print("\n当前全局最佳仍为：")
        print(f"  checkpoint      = {global_best['checkpoint']}")
        print(f"  avg_pc_success  = {global_best['avg_pc_success']:.4f}")
        print(f"  avg_sum_reward  = {global_best['avg_sum_reward']:.4f}")
        print(f"  avg_max_reward  = {global_best['avg_max_reward']:.4f}")

    summary.append(checkpoint_result)

    with open(step_log_path, "w", encoding="utf-8") as f:
        f.write(f"Checkpoint: {step_str}\n\n")
        for r in runs:
            f.write(
                f"Run {r['run_idx']}: "
                f"pc_success={r['pc_success']}, "
                f"avg_sum_reward={r['avg_sum_reward']}, "
                f"avg_max_reward={r['avg_max_reward']}, "
                f"output_dir={r['output_dir']}\n"
            )
        f.write("\n")
        f.write(f"Average pc_success: {avg_pc_success}\n")
        f.write(f"Average avg_sum_reward: {avg_sum_reward}\n")
        f.write(f"Average avg_max_reward: {avg_max_reward}\n\n")
        f.write(
            f"Best run: run_{best_run['run_idx']} "
            f"(pc_success={best_run['pc_success']}, "
            f"avg_sum_reward={best_run['avg_sum_reward']}, "
            f"avg_max_reward={best_run['avg_max_reward']})\n"
        )
        f.write(f"Kept video dir: {kept_video_dir}\n")

    print(f"\nCheckpoint {step_str} 评测完成。")
    print(f"单 checkpoint 汇总日志已保存到：{step_log_path}")

# ===== 保存总表 =====
summary_path = os.path.join(log_dir, "all_checkpoints_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\n" + "=" * 80)
print("全部 checkpoint 评测完成！")
print(f"总汇总文件：{summary_path}")
print(f"最佳视频目录根路径：{best_video_root}")
if global_best is not None:
    print("\n最终全局最佳 checkpoint：")
    print(f"  checkpoint      = {global_best['checkpoint']}")
    print(f"  avg_pc_success  = {global_best['avg_pc_success']:.4f}")
    print(f"  avg_sum_reward  = {global_best['avg_sum_reward']:.4f}")
    print(f"  avg_max_reward  = {global_best['avg_max_reward']:.4f}")
    print(f"  kept_video_dir  = {global_best['kept_video_dir']}")
print("=" * 80)


[1/7] 开始评测 checkpoint: 040000
Checkpoint 040000 not found, skipped.

[2/7] 开始评测 checkpoint: 045000

--- Checkpoint 045000 | Run 1/3 开始 ---
Run 1/3 完成：
  pc_success     = 12.0
  avg_sum_reward = 96.42968390362475
  avg_max_reward = 0.7634090402752011
  output_dir     = outputs/eval/2026-04-09/14-01-24_pusht_act
  >>> 当前 checkpoint 045000 内部最佳 run 已更新为 Run 1

--- Checkpoint 045000 | Run 2/3 开始 ---
Run 2/3 完成：
  pc_success     = 14.000000000000002
  avg_sum_reward = 95.46777891938498
  avg_max_reward = 0.7731624025789166
  output_dir     = outputs/eval/2026-04-09/14-02-49_pusht_act
  >>> 当前 checkpoint 045000 内部最佳 run 已更新为 Run 2

--- Checkpoint 045000 | Run 3/3 开始 ---
Run 3/3 完成：
  pc_success     = 18.0
  avg_sum_reward = 99.12052109513066
  avg_max_reward = 0.7910816926698735
  output_dir     = outputs/eval/2026-04-09/14-04-14_pusht_act
  >>> 当前 checkpoint 045000 内部最佳 run 已更新为 Run 3

--------------------------------------------------------------------------------
Checkpoint 045000 三次平均结果

In [ ]:
import os
from datetime import datetime

test_dir = "/content/drive/MyDrive/lerobot_upload_test"
os.makedirs(test_dir, exist_ok=True)

test_file = os.path.join(test_dir, "drive_test.txt")
with open(test_file, "w", encoding="utf-8") as f:
    f.write("Google Drive upload test success.\n")
    f.write(f"time = {datetime.now()}\n")

print("测试完成。")
print("已写入文件：", test_file)
print("请去 Google Drive 看是否存在这个文件。")

测试完成。
已写入文件： /content/drive/MyDrive/lerobot_upload_test/drive_test.txt
请去 Google Drive 看是否存在这个文件。


In [9]:
from huggingface_hub import notebook_login
notebook_login()

In [10]:
import os
from huggingface_hub import HfApi, create_repo
from datetime import datetime

HF_USERNAME = "LDD11"   # 这里改成你的 Hugging Face 用户名
TEST_REPO = f"{HF_USERNAME}/lerobot-upload-test"

# 1. 创建测试仓库（已存在也没关系）
create_repo(repo_id=TEST_REPO, repo_type="model", exist_ok=True)

# 2. 准备本地测试文件
local_dir = "/content/hf_upload_test"
os.makedirs(local_dir, exist_ok=True)

test_file = os.path.join(local_dir, "hf_test.txt")
with open(test_file, "w", encoding="utf-8") as f:
    f.write("Hugging Face upload test success.\n")
    f.write(f"time = {datetime.now()}\n")

# 3. 上传
api = HfApi()
api.upload_file(
    path_or_fileobj=test_file,
    path_in_repo="docs/hf_test.txt",
    repo_id=TEST_REPO,
    repo_type="model"
)

print("测试完成。")
print("已上传到仓库：", TEST_REPO)
print("仓库内路径：docs/hf_test.txt")

测试完成。
已上传到仓库： LDD11/lerobot-upload-test
仓库内路径：docs/hf_test.txt


In [11]:
import os
from huggingface_hub import HfApi, create_repo
from datetime import datetime

HF_USERNAME = "LDD11"   # 改成你的用户名
TEST_REPO = f"{HF_USERNAME}/lerobot-upload-test-folder"

create_repo(repo_id=TEST_REPO, repo_type="model", exist_ok=True)

local_dir = "/content/hf_folder_test"
os.makedirs(local_dir, exist_ok=True)

for i in range(3):
    with open(os.path.join(local_dir, f"test_{i}.txt"), "w", encoding="utf-8") as f:
        f.write(f"folder upload test {i}\n")
        f.write(f"time = {datetime.now()}\n")

api = HfApi()
api.upload_folder(
    folder_path=local_dir,
    repo_id=TEST_REPO,
    repo_type="model",
    path_in_repo="docs"
)

print("文件夹上传测试完成。")
print("仓库：", TEST_REPO)

文件夹上传测试完成。
仓库： LDD11/lerobot-upload-test-folder


In [20]:
#测试学习率
import os
import re
import json
import shutil
import subprocess
from datetime import datetime
from huggingface_hub import HfApi, create_repo

# =========================================================
# 0. 你主要改这里
# =========================================================
LEROBOT_ROOT = "/content/lerobot"
TRAIN_SCRIPT = os.path.join(LEROBOT_ROOT, "src/lerobot/scripts/lerobot_train.py")

# Google Drive 根目录（新目录会自动创建）
DRIVE_ROOT = "/content/drive/MyDrive/lerobot_autotune"

# 你的 Hugging Face 用户名
HF_USERNAME = "LDD11"

# 基础配置
DATASET_REPO_ID = "lerobot/pusht"
POLICY_TYPE = "act"
DEVICE = "cuda"

CHUNK_SIZE = 50
N_ACTION_STEPS = 18  # 修改为18
USE_AMP = False

# 学习率精细搜索
LR_LIST = [1.2e-5, 1.5e-5, 1.8e-5, 2.2e-5, 2.5e-5, 3e-5]  # 不包括 2e-5

# 训练到 70000，每 2500 保存
TOTAL_STEPS = 70000
SAVE_FREQ = 2500

# 测试：40000 到 70000，每 2500 一个 checkpoint
EVAL_STEPS = list(range(40000, 70001, 2500))
EVAL_REPEATS = 3

LOCAL_RUN_ROOT = os.path.join(LEROBOT_ROOT, "autotune_lr_runs")
LOCAL_RESULT_ROOT = os.path.join(LEROBOT_ROOT, "autotune_lr_results")

EXP_NAME = f"autotune_lr_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

os.makedirs(LOCAL_RUN_ROOT, exist_ok=True)
os.makedirs(LOCAL_RESULT_ROOT, exist_ok=True)
os.makedirs(DRIVE_ROOT, exist_ok=True)

# =========================================================
# 1. 工具函数
# =========================================================
def run_cmd(cmd, cwd=None):
    proc = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    return proc.returncode, proc.stdout

def parse_eval_metrics(text: str):
    def grab(name):
        m = re.search(rf"'{name}':\s*([0-9eE\.\-]+)", text)
        return float(m.group(1)) if m else None

    m_out = re.search(r"Output dir:\s*(.+)", text)
    output_dir = m_out.group(1).strip() if m_out else None

    return {
        "pc_success": grab("pc_success"),
        "avg_sum_reward": grab("avg_sum_reward"),
        "avg_max_reward": grab("avg_max_reward"),
        "eval_ep_s": grab("eval_ep_s"),
        "output_dir": output_dir,
    }

def better_ckpt(a, b):
    if b is None:
        return True
    ka = (a["avg_pc_success"], a["avg_sum_reward"], a["avg_max_reward"])
    kb = (b["avg_pc_success"], b["avg_sum_reward"], b["avg_max_reward"])
    return ka > kb

def better_run(a, b):
    if b is None:
        return True
    ka = (a["pc_success"], a["avg_sum_reward"], a["avg_max_reward"])
    kb = (b["pc_success"], b["avg_sum_reward"], b["avg_max_reward"])
    return ka > kb

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def write_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def copy_checkpoints_and_docs_to_drive(local_run_dir, doc_paths, drive_exp_dir):
    run_name = os.path.basename(local_run_dir)
    dst_run_dir = os.path.join(drive_exp_dir, run_name)
    ensure_dir(dst_run_dir)

    src_ckpt_dir = os.path.join(local_run_dir, "checkpoints")
    dst_ckpt_dir = os.path.join(dst_run_dir, "checkpoints")
    if os.path.isdir(dst_ckpt_dir):
        shutil.rmtree(dst_ckpt_dir)
    if os.path.isdir(src_ckpt_dir):
        shutil.copytree(src_ckpt_dir, dst_ckpt_dir)

    dst_doc_dir = os.path.join(dst_run_dir, "docs")
    ensure_dir(dst_doc_dir)
    for p in doc_paths:
        if os.path.isfile(p):
            shutil.copy2(p, os.path.join(dst_doc_dir, os.path.basename(p)))

    return dst_run_dir

def upload_checkpoints_and_docs_to_hf(local_run_dir, doc_paths, hf_repo_id):
    api = HfApi()
    create_repo(repo_id=hf_repo_id, repo_type="model", exist_ok=True)

    src_ckpt_dir = os.path.join(local_run_dir, "checkpoints")
    if os.path.isdir(src_ckpt_dir):
        api.upload_folder(
            folder_path=src_ckpt_dir,
            repo_id=hf_repo_id,
            repo_type="model",
            path_in_repo="checkpoints"
        )

    tmp_doc_dir = os.path.join("/tmp", f"docs_{os.path.basename(local_run_dir)}")
    if os.path.isdir(tmp_doc_dir):
        shutil.rmtree(tmp_doc_dir)
    os.makedirs(tmp_doc_dir, exist_ok=True)

    for p in doc_paths:
        if os.path.isfile(p):
            shutil.copy2(p, os.path.join(tmp_doc_dir, os.path.basename(p)))

    api.upload_folder(
        folder_path=tmp_doc_dir,
        repo_id=hf_repo_id,
        repo_type="model",
        path_in_repo="docs"
    )

def make_run_name(lr):
    lr_str = str(lr).replace("-", "").replace(".", "")
    return f"pusht_act_chunk50_action20_lr{lr_str}_70000_save2500"

def run_cmd(cmd, cwd=None):
    print(f"Running command: {' '.join(cmd)}")  # 打印命令
    proc = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    print(f"Return code: {proc.returncode}")
    print(f"Stdout: {proc.stdout}")
    print(f"Stderr: {proc.stderr}")  # 打印错误输出
    return proc.returncode, proc.stdout

# =========================================================
# 2. 主流程
# =========================================================
leaderboard = []
global_best = None
drive_exp_dir = os.path.join(DRIVE_ROOT, EXP_NAME)
ensure_dir(drive_exp_dir)

print("=" * 100)
print(f"开始学习率自动调参实验：{EXP_NAME}")
print(f"Drive 输出目录：{drive_exp_dir}")
print("=" * 100)

for idx, lr in enumerate(LR_LIST, start=1):
    run_name = make_run_name(lr)
    local_run_dir = os.path.join(LOCAL_RUN_ROOT, run_name)
    local_result_dir = os.path.join(LOCAL_RESULT_ROOT, run_name)
    ensure_dir(local_result_dir)

    print("\n" + "=" * 100)
    print(f"[{idx}/{len(LR_LIST)}] 开始参数组")
    print(f"run_name         = {run_name}")
    print(f"lr               = {lr}")
    print(f"chunk_size       = {CHUNK_SIZE}")
    print(f"n_action_steps   = {N_ACTION_STEPS}")
    print(f"use_amp          = {USE_AMP}")
    print(f"steps            = {TOTAL_STEPS}")
    print(f"save_freq        = {SAVE_FREQ}")
    print("=" * 100)

    params_doc = {
        "exp_name": EXP_NAME,
        "search_type": "lr_tuning",
        "run_name": run_name,
        "lr": lr,
        "chunk_size": CHUNK_SIZE,
        "n_action_steps": N_ACTION_STEPS,
        "use_amp": USE_AMP,
        "total_steps": TOTAL_STEPS,
        "save_freq": SAVE_FREQ,
        "eval_steps": EVAL_STEPS,
        "eval_repeats": EVAL_REPEATS,
        "dataset_repo_id": DATASET_REPO_ID,
        "policy_type": POLICY_TYPE,
    }
    params_doc_path = os.path.join(local_result_dir, "params.json")
    write_json(params_doc_path, params_doc)

    # ---------- 自动训练 ----------
    train_cmd = [
        "python", TRAIN_SCRIPT,
        f"--dataset.repo_id={DATASET_REPO_ID}",
        f"--policy.type={POLICY_TYPE}",
        f"--output_dir={local_run_dir}",
        f"--job_name={run_name}",
        f"--policy.device={DEVICE}",
        "--wandb.enable=False",
        f"--steps={TOTAL_STEPS}",
        f"--save_freq={SAVE_FREQ}",
        f"--policy.chunk_size={CHUNK_SIZE}",
        f"--policy.n_action_steps={N_ACTION_STEPS}",
        f"--optimizer.lr={lr}",
        f"--policy.optimizer_lr={lr}",
        f"--policy.optimizer_lr_backbone={lr}",
        f"--policy.use_amp={'true' if USE_AMP else 'false'}",
        f"--policy.repo_id={HF_USERNAME}/{run_name}",
    ]

    print("\n>>> 自动开始训练...")
    code, train_out = run_cmd(train_cmd, cwd=LEROBOT_ROOT)

    train_log_path = os.path.join(local_result_dir, "train_log.txt")
    with open(train_log_path, "w", encoding="utf-8") as f:
        f.write(train_out)

    if code != 0:
        print("训练失败，跳过该参数组。")
        leaderboard.append({
            "run_name": run_name,
            "n_action_steps": n_action_steps,
            "status": "train_failed"
        })
        continue

    print(">>> 训练结束，开始自动测试。")

    # ---------- 自动测试 ----------
    per_ckpt_results = []
    best_ckpt = None

    for step in EVAL_STEPS:
        step_str = f"{step:06d}"
        model_path = os.path.join(local_run_dir, "checkpoints", step_str, "pretrained_model")
        if not os.path.isdir(model_path):
            print(f"\nCheckpoint {step_str} 不存在，跳过。")
            continue

        print("\n" + "-" * 90)
        print(f"开始测试 checkpoint = {step_str}")
        print(f"当前参数：lr={lr}, chunk_size={CHUNK_SIZE}, n_action_steps={n_action_steps}, use_amp={USE_AMP}")
        print("-" * 90)

        runs = []
        best_single_run = None

        for rep in range(1, EVAL_REPEATS + 1):
            print(f"\n--- checkpoint {step_str} | run {rep}/{EVAL_REPEATS} ---")

            eval_cmd = [
                "lerobot-eval",
                f"--policy.path={model_path}",
                "--env.type=pusht",
                "--eval.batch_size=50",
                "--eval.n_episodes=50",
                f"--policy.device={DEVICE}",
            ]

            code_eval, eval_out = run_cmd(eval_cmd, cwd=LEROBOT_ROOT)

            eval_log_path = os.path.join(local_result_dir, f"eval_{step_str}_run{rep}.txt")
            with open(eval_log_path, "w", encoding="utf-8") as f:
                f.write(eval_out)

            if code_eval != 0:
                print("本次评测失败。")
                continue

            metrics = parse_eval_metrics(eval_out)
            metrics["run_idx"] = rep
            runs.append(metrics)

            print(f"pc_success     = {metrics['pc_success']}")
            print(f"avg_sum_reward = {metrics['avg_sum_reward']}")
            print(f"avg_max_reward = {metrics['avg_max_reward']}")
            print(f"eval_ep_s      = {metrics['eval_ep_s']}")
            print(f"output_dir     = {metrics['output_dir']}")

            if better_run(metrics, best_single_run):
                best_single_run = metrics
                print(f">>> 当前 checkpoint {step_str} 内部最佳 run 更新为 run {rep}")

        if not runs:
            continue

        avg_pc_success = sum(x["pc_success"] for x in runs) / len(runs)
        avg_sum_reward = sum(x["avg_sum_reward"] for x in runs) / len(runs)
        avg_max_reward = sum(x["avg_max_reward"] for x in runs) / len(runs)
        avg_eval_ep_s = sum(x["eval_ep_s"] for x in runs if x["eval_ep_s"] is not None) / len(runs)

        ckpt_result = {
            "checkpoint": step_str,
            "avg_pc_success": avg_pc_success,
            "avg_sum_reward": avg_sum_reward,
            "avg_max_reward": avg_max_reward,
            "avg_eval_ep_s": avg_eval_ep_s,
            "best_run_idx": best_single_run["run_idx"] if best_single_run else None,
        }
        per_ckpt_results.append(ckpt_result)

        print("\n>>> 三次平均结果：")
        print(f"checkpoint      = {step_str}")
        print(f"avg_pc_success  = {avg_pc_success:.4f}")
        print(f"avg_sum_reward  = {avg_sum_reward:.4f}")
        print(f"avg_max_reward  = {avg_max_reward:.4f}")
        print(f"avg_eval_ep_s   = {avg_eval_ep_s:.4f}")

        if better_ckpt(ckpt_result, best_ckpt):
            best_ckpt = ckpt_result
            print(f">>> 当前参数组最佳 checkpoint 更新为：{step_str}")

    if best_ckpt is None:
        leaderboard.append({
            "run_name": run_name,
            "n_action_steps": n_action_steps,
            "status": "eval_failed"
        })
        continue

    run_summary = {
        "run_name": run_name,
        "search_type": "n_action_steps_tuning",
        "lr": LR,
        "chunk_size": CHUNK_SIZE,
        "n_action_steps": n_action_steps,
        "use_amp": USE_AMP,
        "best_checkpoint": best_ckpt["checkpoint"],
        "avg_pc_success": best_ckpt["avg_pc_success"],
        "avg_sum_reward": best_ckpt["avg_sum_reward"],
        "avg_max_reward": best_ckpt["avg_max_reward"],
        "avg_eval_ep_s": best_ckpt["avg_eval_ep_s"],
        "all_checkpoint_results": per_ckpt_results,
        "status": "ok"
    }

    run_summary_path = os.path.join(local_result_dir, "run_summary.json")
    write_json(run_summary_path, run_summary)

    print("\n当前参数组最佳结果：")
    print(json.dumps(run_summary, indent=2, ensure_ascii=False))

    if global_best is None or better_ckpt(
        {
            "avg_pc_success": run_summary["avg_pc_success"],
            "avg_sum_reward": run_summary["avg_sum_reward"],
            "avg_max_reward": run_summary["avg_max_reward"],
        },
        {
            "avg_pc_success": global_best["avg_pc_success"],
            "avg_sum_reward": global_best["avg_sum_reward"],
            "avg_max_reward": global_best["avg_max_reward"],
        }
    ):
        global_best = run_summary
        print("\n" + "🔥" * 30)
        print("发现新的全局最佳参数组！")
        print(json.dumps(global_best, indent=2, ensure_ascii=False))
        print("🔥" * 30)

    # ---------- 上传到 Google Drive ----------
    print("\n>>> 开始上传到 Google Drive（仅 checkpoints + 参数文档）...")
    copy_checkpoints_and_docs_to_drive(
        local_run_dir=local_run_dir,
        doc_paths=[params_doc_path, train_log_path, run_summary_path],
        drive_exp_dir=drive_exp_dir
    )
    print(">>> Google Drive 上传完成。")

    # ---------- 上传到 Hugging Face ----------
    print("\n>>> 开始上传到 Hugging Face（仅 checkpoints + 参数文档）...")
    upload_checkpoints_and_docs_to_hf(
        local_run_dir=local_run_dir,
        doc_paths=[params_doc_path, train_log_path, run_summary_path],
        hf_repo_id=f"{HF_USERNAME}/{run_name}"
    )
    print(">>> Hugging Face 上传完成。")

    leaderboard.append(run_summary)

valid_results = [x for x in leaderboard if x.get("status") == "ok"]
valid_results.sort(
    key=lambda x: (x["avg_pc_success"], x["avg_sum_reward"], x["avg_max_reward"]),
    reverse=True
)

leaderboard_path = os.path.join(LOCAL_RESULT_ROOT, f"{EXP_NAME}_leaderboard.json")
write_json(leaderboard_path, valid_results)

print("\n" + "=" * 100)
print("n_action_steps 自动调参全部完成。")
print(f"排行榜保存到：{leaderboard_path}")
if valid_results:
    print("\n最终全局最佳参数组：")
    print(json.dumps(valid_results[0], indent=2, ensure_ascii=False))
print("=" * 100)

开始学习率自动调参实验：autotune_lr_20260409_170355
Drive 输出目录：/content/drive/MyDrive/lerobot_autotune/autotune_lr_20260409_170355

[1/6] 开始参数组
run_name         = pusht_act_chunk50_action20_lr12e05_70000_save2500
lr               = 1.2e-05
chunk_size       = 50
n_action_steps   = 18
use_amp          = False
steps            = 70000
save_freq        = 2500

>>> 自动开始训练...
Running command: python /content/lerobot/src/lerobot/scripts/lerobot_train.py --dataset.repo_id=lerobot/pusht --policy.type=act --output_dir=/content/lerobot/autotune_lr_runs/pusht_act_chunk50_action20_lr12e05_70000_save2500 --job_name=pusht_act_chunk50_action20_lr12e05_70000_save2500 --policy.device=cuda --wandb.enable=False --steps=70000 --save_freq=2500 --policy.chunk_size=50 --policy.n_action_steps=18 --optimizer.lr=1.2e-05 --policy.optimizer_lr=1.2e-05 --policy.optimizer_lr_backbone=1.2e-05 --policy.use_amp=false --policy.repo_id=LDD11/pusht_act_chunk50_action20_lr12e05_70000_save2500
Return code: 2
Stdout: 
Stderr: python: ca

In [18]:
# 挂载 Google Drive
from google.colab import drive
drive.mount('/content/gdrive')



Mounted at /content/gdrive


In [61]:
# 登录 Hugging Face
from huggingface_hub import notebook_login
notebook_login()

实验结果总结
本轮实验主要比较了三类配置：

基线 `chunk_size=50, n_action_steps=50` 的表现较差。50 局正式评估中，`40000 step` 成功率为 `6.0%（3/50）`，`80000 step` 仍为 `6.0%（3/50）`，`100000 step` 也还是 `6.0%（3/50）`，说明单纯延长训练步数并没有显著提升最终成功率。

将两者同步缩短为 `chunk_size=20, n_action_steps=20` 后，性能明显改善。`40000 step` 时成功率提升到 `16.0%（8/50）`，但继续训练到 `80000 step` 后成功率下降到 `12.0%（6/50）`。这说明较短动作块更适合 Push-T 任务，但训练更久不一定更好。

效果最好的配置是 `chunk_size=50, n_action_steps=20`。在 `40000 step` 时，50 局正式评估成功率达到 `24.0%（12/50）`，明显优于前两组；继续训练到 `80000 step` 后，成功率仍保持 `24.0%（12/50）`，虽然没有继续上升，但 `avg_sum_reward` 从约 `93.36` 提高到约 `95.97`，`avg_max_reward` 从约 `0.7664` 提高到约 `0.8357`，说明整体行为质量有所提升。

综合来看，本轮实验表明：**“长预测 + 短执行”** 的策略最适合当前任务，即保留较长动作规划能力，同时提高重规划频率，可以显著提升成功率；而单纯增加训练步数，对成功率提升作用有限。


## Login into Hugging Face Hub.
### Now after training is done login into the Hugging Face hub and upload the last checkpoint.

In [ ]:
# from huggingface_hub import HfApi

# HF_USERNAME = "LDD11"
# HF_REPO_NAME = "act-configs"

# api = HfApi()
# repo_id = f"{HF_USERNAME}/{HF_REPO_NAME}"
# files_in_repo = api.list_repo_files(repo_id=repo_id)

# print(f"Files in {repo_id}:")
# for file in files_in_repo:
#     print(f"- {file}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-69d5d23c-5f3756a11e79b9f9600c234f;b6c88078-f25b-4f79-9fea-42e797c5fb3e)

Repository Not Found for url: https://huggingface.co/api/models/LDD11/act-configs/tree/main?recursive=true&expand=false.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.

### Configure Hugging Face Token

Add your HF_TOKEN (AKA Secret) to Google Colab to enable Colab to access your HF repositories. This is optional and might need modification if you are using another cloud provider.

In [ ]:
# from google.colab import userdata
# import os

# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# # Verify token is loaded (optional)
# if os.getenv("HF_TOKEN"):
#     print("Hugging Face token loaded successfully.")
# else:
#     print("Error: Hugging Face token not found. Please set it as a Colab secret named 'HF_TOKEN'.")

SecretNotFoundError: Secret HF_TOKEN does not exist.

In [ ]:
# from google.colab import userdata
# userdata.get('HF_TOKEN')

### Download files from Hugging Face Hub into Colab

Now, you can use `hf_hub_download` to pull the files directly to your Colab environment. You can then download them from Colab to your local machine. This is optional.

In [ ]:
# from huggingface_hub import hf_hub_download

# # Your Hugging Face repository details
# HF_CONFIG_REPO_ID = "${LDD11}/act-configs" # Where train_config.json should be
# HF_POLICY_REPO_ID = "${LDD11}/hf_act_recordpolicy0" # Where the trained model is

# # Define the files to download
# config_file_name = "train_config.json"
# model_file_name = "model.safetensors"
# tokenizer_processor_file_name = "tokenizer_processor.safetensors"

# # Download train_config.json
# train_config_path = hf_hub_download(repo_id=HF_CONFIG_REPO_ID, filename=config_file_name)
# print(f"Downloaded {config_file_name} to: {train_config_path}")

# # Download the trained model files
# model_path = hf_hub_download(repo_id=HF_POLICY_REPO_ID, filename=model_file_name)
# print(f"Downloaded {model_file_name} to: {model_path}")

# tokenizer_processor_path = hf_hub_download(repo_id=HF_POLICY_REPO_ID, filename=tokenizer_processor_file_name)
# print(f"Downloaded {tokenizer_processor_file_name} to: {tokenizer_processor_path}")

# print("\nAll specified files have been downloaded to your Colab environment.")
# print("You can find them in the paths printed above. To download them to your local machine, right-click on the files in the Colab file browser (left sidebar) and select 'Download'.")

### Verify `train_config.json` existence locally. This is needed for restarting training and testing of the policy.

In [ ]:
# !ls -l /content/lerobot/outputs/train/hf_act_record0/

ls: cannot access '/content/lerobot/outputs/train/hf_act_record0/': No such file or directory


In [ ]:
# HF_USERNAME = "${LDD11}"
# HF_REPO_NAME = "act-configs"

# !hf repo-files $HF_USERNAME/$HF_REPO_NAME

In [ ]:
# !hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) 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
Invalid input. Must be one of ('y', 'yes', '1', 'n', 'no', '0', '')
Add token as git credential? (Y/n) y
Traceback (most recent call last):
  File "/usr

In [ ]:
# !hf upload ${LDD11}/hf_act_record0 \
#   /content/lerobot/outputs/train/hf_act_record0/checkpoints/last/pretrained_model

In [ ]:
# !hf auth login

### Create a new repository on Hugging Face Hub

This command will create a new repository under your Hugging Face account.

In [ ]:
# HF_USERNAME = "${LDD11}"
# HF_REPO_NAME = "act-configs"

# !hf repo create $HF_REPO_NAME --type model --private --organization $HF_USERNAME

usage: hf <command> [<args>]
hf: error: unrecognized arguments: --type model --organization


In [ ]:
# ### Upload `train_config.json` to the new repository
# HF_USERNAME = "${LDD11}"
# HF_REPO_NAME = "act-configs"
# LOCAL_CONFIG_PATH = "/content/lerobot/outputs/train/hf_act_record0/train_config.json"

# !hf upload $HF_USERNAME/$HF_REPO_NAME "$LOCAL_CONFIG_PATH" train_config.json

Traceback (most recent call last):
  File "/usr/local/bin/hf", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.11/site-packages/huggingface_hub/cli/hf.py", line 59, in main
    service.run()
  File "/usr/local/lib/python3.11/site-packages/huggingface_hub/cli/upload.py", line 209, in run
    print(self._upload())
          ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/huggingface_hub/cli/upload.py", line 269, in _upload
    raise FileNotFoundError(f"No such file or directory: '{self.local_path}'.")
FileNotFoundError: No such file or directory: '/content/lerobot/outputs/train/hf_act_record0/train_config.json'.
